# C5: SBERT + Ward, provera sa kategorijom (BBC arhiva 2004 do 2005)
**Tip:** vežba iz tvoje pripreme (PDF `sbert_ward`). Dugi članci, pa skraćivanje zbog ograničenja SBERT-a, i provera sa stvarnom kategorijom.

### Zadatak
U fajlu `bbc-news-data.csv` (razdvojen tabom) su BBC članci sa kolonama `category`, `filename`, `title`, `content`. Uzorak od 1500 članaka.
- Napraviti `article_text` od naslova i teksta. Ispitati dužinu (statistika i histogram), pa odlučiti na koliko reči skratiti s obzirom na ograničenje ulaza SBERT-a i obrazložiti.
- Vektori pomoću **all-MiniLM-L12-v2** (lokalno). Objasniti zašto se SBERT-u daje tekst bez čišćenja.
- Kosinusne distance, Ward, dendrogram. Broj klastera na osnovu dendrograma i siluete za 2 do 8, obrazložiti. Veličine klastera.
- Ključne reči: tekst klastera očistiti spaCy-jem, TF-IDF, top 10, grafički prikaz.
- Ime za svaki klaster, da li su razdvojeni.
- Tek na kraju uporediti sa `category` kroz `pd.crosstab` i objasniti neslaganja.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import pdist

import matplotlib.pyplot as plt
import seaborn as sb
from pypalettes import load_cmap

from pathlib import Path
from IPython.display import display

import spacy
from sentence_transformers import SentenceTransformer

RAND_STATE = 1
DATA_DIR = Path.cwd().parent / 'data'
MODELS_DIR = Path.cwd().parent / 'models'

### Šta je urađeno
- Učitani podaci (`sep='\t'`), provereni prazni i duplikati, uzet uzorak od 1500 članaka.
- Napravljena `article_text` (naslov + tačka + tekst).
- Dužina: prosek oko 380 reči, medijana oko 330 (statistika i histogram).
- SBERT model čita najviše 128 tokena (oko 100 reči), pa su članci skraćeni na prvih 120 reči (`article_text_reduced`). Ostatak bi model ionako odsekao.
- Kolona `category` se **ne koristi do poslednjeg koraka**.

In [ ]:
data = pd.read_csv(DATA_DIR / 'bbc-news-data.csv', sep='\t')
data.info()

In [ ]:
data = data.dropna(subset=['content'])
print('Duplikati teksta:', data.duplicated(subset='content').sum())
data = data.drop_duplicates(subset='content')
data = data.sample(n=1500, random_state=RAND_STATE).reset_index(drop=True)

data['article_text'] = data['title'] + '. ' + data['content'].str.strip()

In [ ]:
data['article_len_words'] = data['article_text'].str.split().str.len()
print(data['article_len_words'].describe())

plt.figure(figsize=(8, 5))
plt.hist(data['article_len_words'], bins=50)
plt.title('Duzina clanaka (broj reci)')
plt.show()

data['article_text_reduced'] = data['article_text'].str.split().str[:120].str.join(' ')

---
# ▶ OVDE KREĆEŠ

Sve iznad je gotovo. Od ove ćelije pa nadalje kucaš sam: kod, komentare u markdown ćelijama i tumačenja.

**Tvoji koraci:**

1. `SentenceTransformer(str(MODELS_DIR / 'all-MiniLM-L12-v2'), local_files_only=True)`, proveri `model.max_seq_length`
2. `encode(data['article_text_reduced'].to_numpy(), normalize_embeddings=True)` (n x 384)
3. Linkage (`pdist`, Ward), dendrogram, silueta 2 do 8, izbor i veličine
4. Ključne reči: spaCy `en_core_web_sm` čišćenje u `TfidfVectorizer`, top 10, grafik
5. Imena klastera, pa `pd.crosstab(data['category'], data['clust'], margins=True)` i komentar

---
## Dati kod
Ovo dobijaš uz zadatak.

In [ ]:
# DATI KOD (dobija se na kolokvijumu): prikaz kljucnih reci po klasterima
# Funkciju get_cluster_keywords(cluster_id) pises sam; treba da vrati listu parova (rec, skor).
# Raskomentarisi kad stignes do ovog koraka.

# fig, ax_grid = plt.subplots(nrows=2, ncols=3, figsize=(16, 10), constrained_layout=True)
# axes = ax_grid.flatten()
# cmap = load_cmap("excel_Median")
#
# n_clust = data['clust'].nunique()
# for i in range(n_clust):
#     scored_kwds = get_cluster_keywords(i+1)
#     scored_kwds = sorted(scored_kwds, key=lambda item: item[1])
#     scores = [s for kw, s in scored_kwds]
#     keywords = [kw for kw, s in scored_kwds]
#     axes[i].barh(keywords, scores, color=cmap(i))
#     axes[i].set_title(f"Cluster {i+1}", color=cmap(i))
#     axes[i].grid(visible=True, axis='y', color='gray', alpha=0.35)
#
# for ax in axes[n_clust:]:   # brise prazna polja ako je mreza veca od broja klastera
#     ax.remove()
#
# fig.suptitle("Kljucne reci po klasterima")
# plt.show()

In [ ]:
# DATI KOD (primer): skraceni dendrogram, ako je pun nepregledan
# plt.figure(figsize=(12, 6))
# sch.dendrogram(ward_linkage, truncate_mode='lastp', p=30)
# plt.title('Dendrogram (Ward)')
# plt.ylabel('Distanca')
# plt.show()